In [1]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print('Loading earthquake dataset...')
df = pd.read_csv('../data/earthquakes/usgs_earthquakes.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["time"].min()} to {df["time"].max()}')
print(f'Magnitude range: {df["mag"].min()} to {df["mag"].max()}')
print(f'Columns: {list(df.columns[:8])}')
df.head()

Loading earthquake dataset...
Shape: (8678, 22)
Date range: 2020-01-01 00:28:20.289000+00:00 to 2024-12-30 05:49:02.808000+00:00
Magnitude range: 5.0 to 8.2
Columns: ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap']


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2020-01-01 00:28:20.289000+00:00,-5.3245,152.5514,40.24,5.1,mb,NaN,74.0,1.190,0.79,...,2025-12-22T18:25:00.644Z,"112 km SSE of Kokopo, Papua New Guinea",earthquake,9.4,6.3,0.044,171.0,reviewed,us,us
1,2020-01-01 03:53:29.023000+00:00,52.6367,159.1614,58.46,5.0,mb,NaN,94.0,0.496,0.66,...,2025-12-22T17:56:37.541Z,"56 km SE of Petropavlovsk-Kamchatsky, Russia",earthquake,7.7,4.6,0.026,492.0,reviewed,us,us
2,2020-01-01 08:31:24.136000+00:00,-17.4280,-172.5044,10.00,5.4,mww,NaN,40.0,2.947,1.44,...,2025-12-22T18:25:01.931Z,"207 km NE of Neiafu, Tonga",earthquake,8.8,1.8,0.086,13.0,reviewed,us,us
3,2020-01-01 16:34:59.178000+00:00,-53.0907,9.5001,10.00,5.0,mb,NaN,105.0,19.424,0.83,...,2020-03-21T17:13:21.040Z,southwest of Africa,earthquake,14.0,1.9,0.082,48.0,reviewed,us,us
4,2020-01-01 16:51:32.285000+00:00,-30.2857,-71.5582,40.88,5.0,mwr,NaN,117.0,0.392,0.72,...,2022-08-08T17:51:20.636Z,"42 km SSW of Coquimbo, Chile",earthquake,5.0,2.5,0.038,67.0,reviewed,us,us


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Prepare features
features = ['latitude', 'longitude', 'depth', 'mag']
df_clean = df[features].dropna().reset_index(drop=True)

# Normalize
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df_clean)

# Build sequences - use last 10 earthquakes to predict next magnitude
SEQ_LEN = 10
X_seq, y_seq = [], []

for i in range(SEQ_LEN, len(scaled)):
    X_seq.append(scaled[i-SEQ_LEN:i])
    y_seq.append(scaled[i, 3])  # magnitude is index 3

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Train/test split
split = int(len(X_seq) * 0.8)
X_train = torch.FloatTensor(X_seq[:split])
y_train = torch.FloatTensor(y_seq[:split])
X_test = torch.FloatTensor(X_seq[split:])
y_test = torch.FloatTensor(y_seq[split:])

print(f'Sequence shape: {X_seq.shape}')
print(f'Training sequences: {X_train.shape}')
print(f'Test sequences: {X_test.shape}')
print('Data ready for LSTM and TFT')

Sequence shape: (8668, 10, 4)
Training sequences: torch.Size([6934, 10, 4])
Test sequences: torch.Size([1734, 10, 4])
Data ready for LSTM and TFT


In [3]:
# LSTM Model Definition
class LSTMModel(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

# Train LSTM
print('Training LSTM...')
start = time.time()

lstm_model = LSTMModel()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

lstm_model.train()
for epoch in range(30):
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = lstm_model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'  Epoch {epoch+1}/30 | Loss: {total_loss/len(train_loader):.4f}')

lstm_time = time.time() - start

# Evaluate
lstm_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(X_test).numpy()

lstm_rmse = np.sqrt(mean_squared_error(y_test.numpy(), lstm_preds))
lstm_mae = mean_absolute_error(y_test.numpy(), lstm_preds)

print(f'\nLSTM Results:')
print(f'  RMSE: {lstm_rmse:.4f}')
print(f'  MAE:  {lstm_mae:.4f}')
print(f'  Time: {lstm_time:.1f}s')

Training LSTM...
  Epoch 10/30 | Loss: 0.0157
  Epoch 20/30 | Loss: 0.0156
  Epoch 30/30 | Loss: 0.0156

LSTM Results:
  RMSE: 0.1192
  MAE:  0.0926
  Time: 13.7s


In [4]:
# Temporal Fusion Transformer - simplified implementation
class TFTModel(nn.Module):
    def __init__(self, input_size=4, d_model=64, n_heads=4, seq_len=10):
        super(TFTModel, self).__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.positional = nn.Parameter(torch.randn(seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, 
            dim_feedforward=128, dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.attention_gate = nn.Linear(d_model, d_model)
        self.output = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional
        x = self.transformer(x)
        # Gated attention - key TFT component
        gate = torch.sigmoid(self.attention_gate(x))
        x = gate * x
        return self.output(x[:, -1, :]).squeeze()

# Train TFT
print('Training Temporal Fusion Transformer (Google, 2021)...')
start = time.time()

tft_model = TFTModel()
optimizer = torch.optim.Adam(tft_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

tft_model.train()
for epoch in range(30):
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = tft_model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'  Epoch {epoch+1}/30 | Loss: {total_loss/len(train_loader):.4f}')

tft_time = time.time() - start

# Evaluate
tft_model.eval()
with torch.no_grad():
    tft_preds = tft_model(X_test).numpy()

tft_rmse = np.sqrt(mean_squared_error(y_test.numpy(), tft_preds))
tft_mae = mean_absolute_error(y_test.numpy(), tft_preds)

print(f'\nTFT Results:')
print(f'  RMSE: {tft_rmse:.4f}')
print(f'  MAE:  {tft_mae:.4f}')
print(f'  Time: {tft_time:.1f}s')

# Final comparison
print('\n' + '='*45)
print(f'{"MODEL":<25} {"RMSE":>8} {"MAE":>8} {"TIME":>8}')
print('='*45)

models = {
    'LSTM (baseline)': (lstm_rmse, lstm_mae, lstm_time),
    'TFT - Google 2021': (tft_rmse, tft_mae, tft_time)
}

winner = min(models, key=lambda x: models[x][0])
for model, (rmse, mae, t) in models.items():
    marker = ' <-- winner' if model == winner else ''
    print(f'{model:<25} {rmse:>8.4f} {mae:>8.4f} {t:>7.1f}s{marker}')
print('='*45)
print('\nMetric: lower RMSE = better earthquake magnitude prediction')

Training Temporal Fusion Transformer (Google, 2021)...
  Epoch 10/30 | Loss: 0.0159
  Epoch 20/30 | Loss: 0.0158
  Epoch 30/30 | Loss: 0.0157

TFT Results:
  RMSE: 0.1179
  MAE:  0.0902
  Time: 26.1s

MODEL                         RMSE      MAE     TIME
LSTM (baseline)             0.1192   0.0926    13.7s
TFT - Google 2021           0.1179   0.0902    26.1s <-- winner

Metric: lower RMSE = better earthquake magnitude prediction


In [5]:
# Save earthquake models from this notebook
import joblib
import torch
import os

os.makedirs('../models', exist_ok=True)

# Earthquake models - from this notebook
torch.save(tft_model.state_dict(), '../models/tft_earthquake.pt')
joblib.dump(scaler, '../models/earthquake_scaler.joblib', compress=3)
print('Earthquake models saved')

Earthquake models saved
